In [ ]:
#%pip install rapidfuzz

In [25]:
#Reconciliation Automation using `pandas` and `rapidfuzz` to match transactions between
#bank and internal records, flagging unmatched items or close matches.

In [1]:
import pandas as pd
from rapidfuzz import fuzz, process


In [9]:
### **Use Case**:  
#Two CSV files:  
#- `bank_records.csv`  
#- `internal_records.csv`  
#Each file has:
#- `Date`, `Amount`, `Description`

In [27]:
def load_data(bank_records, internal_records):
    bank_df = pd.read_csv(bank_records)
    internal_df = pd.read_csv(internal_records)
    return (bank_df, internal_df)

In [43]:
%%capture
load_data("bank_records.csv","internal_records.csv")

In [11]:
# Normalize descriptions for better matchinga
bank_df['Norm_Desc'] = bank_df['Description'].str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True)
internal_df['Norm_Desc'] = internal_df['Description'].str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True)

In [37]:
def normalize_data(bank_df, internal_df):
    bank_df['Norm_Desc'] = bank_df['Description'].str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True)
    internal_df['Norm_Desc'] = internal_df['Description'].str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True)
    return (bank_df, internal_df)

In [41]:
%%capture
normalize_data(bank_df, internal_df)

In [13]:
# Store matches and mismatches
matches = []
unmatched_bank = []

In [45]:
def create_lists(matches, unmatched_bank):
    matches = []
    unmatched_bank = []
    return (matches, unmatched_bank)

In [65]:
create_lists(matches, unmatched_bank)

([], [])

In [59]:
def reconcile(bank_df, internal_df):
    # Reconciliation Logic
    for idx, bank_row in bank_df.iterrows():
    # Try to match by amount and fuzzy description
        candidates = internal_df[internal_df['Amount'] == bank_row['Amount']]
    
        if not candidates.empty:
            best_match = process.extractOne(
                bank_row['Norm_Desc'],
                candidates['Norm_Desc'],
                scorer=fuzz.token_sort_ratio
            )
        
            if best_match and best_match[1] >= 85: # similarity threshold
                match_idx = candidates[candidates['Norm_Desc'] == best_match[0]].index[0]
                matches.append({
                    'Bank_Date': bank_row['Date'],
                    'Bank_Amount': bank_row['Amount'],
                    'Bank_Desc': bank_row['Description'],
                    'Internal_Date': internal_df.loc[match_idx, 'Date'],
                    'Internal_Desc': internal_df.loc[match_idx, 'Description'],
                    'Score': best_match[1]
                })
                internal_df.drop(match_idx, inplace=True) # Remove matched row
            else:
                unmatched_bank.append(bank_row)
        else:
            unmatched_bank.append(bank_row)

    # Convert to DataFrames
    matched_df = pd.DataFrame(matches)
    unmatched_bank_df = pd.DataFrame(unmatched_bank)
    
    return (matched_df, unmatched_bank_df)


In [63]:
%%capture
reconcile(bank_df, internal_df)

In [67]:
%%capture
load_data("bank_records.csv","internal_records.csv")
normalize_data(bank_df, internal_df)
create_lists(matches, unmatched_bank)
reconcile(bank_df, internal_df)

In [73]:
def run():
    load_data("bank_records.csv","internal_records.csv")
    normalize_data(bank_df, internal_df)
    create_lists(matches, unmatched_bank)
    reconcile(bank_df, internal_df)
    return (matched_df, unmatched_bank_df)

In [77]:
%%capture
run()

In [81]:
matched_df

#unmatched_bank_df.head()

,Bank_Date,Bank_Amount,Bank_Desc,Internal_Date,Internal_Desc,Score
0,1/1/2025,100,A,1/1/2025,A,100.0
1,1/2/2025,200,B,1/2/2025,B,100.0
2,1/5/2025,500,E,1/5/2025,E,100.0
3,1/7/2025,700,G,1/7/2025,G,100.0
4,1/10/2025,1000,J,1/10/2025,J,100.0
5,1/11/2025,1100,K,1/11/2025,K,100.0
6,1/13/2025,1300,M,1/13/2025,M,100.0
7,1/14/2025,1400,N,1/14/2025,N,100.0
8,1/17/2025,1700,Q,1/17/2025,Q,100.0
9,1/18/2025,1800,R,1/18/2025,R,100.0


In [5]:
# Load data
bank_df = pd.read_csv("bank_records.csv")
internal_df = pd.read_csv("internal_records.csv")

In [53]:

# Reconciliation Logic
for idx, bank_row in bank_df.iterrows():
    # Try to match by amount and fuzzy description
    candidates = internal_df[internal_df['Amount'] == bank_row['Amount']]
    
    if not candidates.empty:
        best_match = process.extractOne(
            bank_row['Norm_Desc'],
            candidates['Norm_Desc'],
            scorer=fuzz.token_sort_ratio
        )
        
        if best_match and best_match[1] >= 85: # similarity threshold
            match_idx = candidates[candidates['Norm_Desc'] == best_match[0]].index[0]
            matches.append({
                'Bank_Date': bank_row['Date'],
                'Bank_Amount': bank_row['Amount'],
                'Bank_Desc': bank_row['Description'],
                'Internal_Date': internal_df.loc[match_idx, 'Date'],
                'Internal_Desc': internal_df.loc[match_idx, 'Description'],
                'Score': best_match[1]
            })
            internal_df.drop(match_idx, inplace=True) # Remove matched row
        else:
            unmatched_bank.append(bank_row)
    else:
        unmatched_bank.append(bank_row)

    # Convert to DataFrames
    matched_df = pd.DataFrame(matches)
    unmatched_bank_df = pd.DataFrame(unmatched_bank)


SyntaxError: 'return' outside function (<ipython-input-53-275c0921910c>, line 33)

In [ ]:


# Save results
matched_df.to_csv("matched_records.csv", index=False)
unmatched_bank_df.to_csv("unmatched_bank_records.csv", index=False)

print(f"Reconciliation completed.")
print(f"Matched: {len(matched_df)} records")
print(f"Unmatched Bank Records: {len(unmatched_bank_df)}")
```

---